# Notebook C -- InternVL3-8B re-runs at max_new_tokens=64 + InternVL3-2B probe
Order: text_only_v2 (8B, 64 tok) -> conversion_test (8B, 64 tok) -> aggregates -> cap_check -> probe (InternVL3-2B, data_balanced).

Why: in notebook B, 166 of 2016 text-only and 79 of 1728 conversion replies of InternVL3-8B hit the 32-token cap (a verbose
explanation was cut off before the time). Qwen-3B, Qwen-7B and InternVL3-2B had ZERO cap hits, so they are NOT re-run.
InternVL3-8B describe_check replies never exceeded 50 tokens (cap 120), so it is not re-run either.

## Inputs (attach these as Kaggle datasets: Add Input -> your datasets)
| Dataset (name is free) | Contents at the dataset ROOT |
|---|---|
| **clock-code** | `adapters.py`, `clocks.py`, `eval_behavior.py`, `text_only_v2.py`, `conversion_test.py`, `probe.py` (UPDATED), `cap_check.py` (NEW) |
| **clock-data** | folder `data_balanced/` (480 clocks `probe_0000.png`... + `data.csv`) |

(`describe_check.py`, `data/` and `data_positions/` are NOT needed here; `conversion_test.py` draws its own images.)

**Settings:** Accelerator = GPU T4 x2 - Internet = ON - then **Save Version -> Save & Run All (Commit)**. Do not upgrade transformers.

**Output:** one file, `internvl3_rerun_probe_results.zip` (refreshed after every step). A failing step is recorded and the notebook
continues; see `STATUS.txt` inside the zip. The zip includes the probe's saved activations (~170 MB) so later analysis needs no GPU.

Rough runtime (a GUESS, not measured): re-runs ~1-1.5 h (replies are longer at 64 tokens), probe extraction ~10-20 min,
probe fitting (CPU) ~20-60 min; no new step starts after 10.5 h.


In [ ]:
# CELL 1 -- GPU CHECK: stop the notebook right here if CUDA is not available
import torch, subprocess

MIN_GPUS = 2
if not torch.cuda.is_available():
    raise RuntimeError("STOP: CUDA is not available. Settings -> Accelerator -> 'GPU T4 x2', then Save & Run All again.")
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory / 2**30:.1f} GiB")
if n_gpu < MIN_GPUS:
    raise RuntimeError(f"STOP: this notebook needs {MIN_GPUS} GPU(s) but only {n_gpu} visible. Choose 'GPU T4 x2'.")
subprocess.run(["nvidia-smi"])
print("GPU check passed.")


In [ ]:
# CELL 2 -- COPY INPUTS from the attached Kaggle datasets into a clean working folder
import os, shutil, sys, time, json, subprocess, zipfile, datetime, hashlib

T_START = time.time()
INPUT, WORK = "/kaggle/input", "/kaggle/working"
RUN = f"{WORK}/run"
OUT_ZIP = f"{WORK}/internvl3_rerun_probe_results.zip"
CODE_FILES = ["adapters.py", "clocks.py", "eval_behavior.py", "text_only_v2.py", "conversion_test.py", "probe.py", "cap_check.py"]
EXPECT_N_BAL = 480                              # images in data_balanced/

shutil.rmtree(RUN, ignore_errors=True)
os.makedirs(f"{RUN}/logs")

def find_root(marker, maxdepth=4):
    """Directory under /kaggle/input that contains `marker` (a relative path). Dataset slugs don't matter."""
    base = INPUT.rstrip("/").count("/")
    for dirpath, dirnames, _ in os.walk(INPUT):
        if dirpath.count("/") - base > maxdepth:
            dirnames[:] = []
            continue
        if os.path.exists(os.path.join(dirpath, marker)):
            return dirpath
    raise FileNotFoundError(f"Could not find '{marker}' under {INPUT}. Attach the dataset that contains it "
                            f"(see the checklist at the top of this notebook).")

code_root = find_root("adapters.py")
for f in CODE_FILES:
    shutil.copy(os.path.join(code_root, f), RUN)
print("code from:", code_root)

for name, expect in (("data_balanced", EXPECT_N_BAL),):
    root = find_root(f"{name}/data.csv")
    shutil.copytree(os.path.join(root, name), f"{RUN}/{name}")
    n_png = len([x for x in os.listdir(f"{RUN}/{name}") if x.endswith(".png")])
    sha = hashlib.sha256(open(f"{RUN}/{name}/data.csv", "rb").read()).hexdigest()[:12]
    print(f"{name}/: {n_png} PNGs (expected {expect}), data.csv sha256[:12]={sha}, from {root}")
    assert n_png == expect, f"{name}/ has {n_png} PNGs, expected {expect} -- wrong or incomplete dataset"
print("inputs copied OK.")


In [ ]:
# CELL 3 -- ENVIRONMENT: env vars, versions, internet check (model weights are downloaded from Hugging Face)
os.environ.update({"PYTHONUNBUFFERED": "1", "TQDM_MININTERVAL": "60", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
                   "TOKENIZERS_PARALLELISM": "false", "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1"})
r = subprocess.run([sys.executable, "-c",
    "import torch, transformers, accelerate, torchvision, matplotlib, pandas, PIL, sys;"
    "print('python', sys.version.split()[0]);"
    "[print(m.__name__, m.__version__) for m in (torch, transformers, accelerate, torchvision, matplotlib, pandas, PIL)];"
    "print('cuda', torch.version.cuda, '| gpus', torch.cuda.device_count())"],
    capture_output=True, text=True)
print(r.stdout, r.stderr[-800:])
assert r.returncode == 0, "a required package is missing -- see the error above"
open(f"{RUN}/env.txt", "w").write(r.stdout + "\nstarted " + datetime.datetime.utcnow().isoformat() + "Z\n")
import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=20)
    print("internet OK (huggingface.co reachable)")
except Exception as e:
    raise RuntimeError(f"STOP: huggingface.co not reachable ({e}). Settings -> Internet -> ON.")


In [ ]:
# CELL 4 -- THE STEPS (re-runs first, probe last)
MODEL_8B = "OpenGVLab/InternVL3-8B-hf"
MODEL_2B = "OpenGVLab/InternVL3-2B-hf"
MAXTOK = "64"          # was 32 in notebook B
DEADLINE_H = 10.5      # do not START a new step after this many hours (Kaggle sessions end at 12 h)

STEPS = [
    ("1_text_only_v2_8b_64tok",    ["python", "text_only_v2.py", "--model_id", MODEL_8B, "--max_new_tokens", MAXTOK]),
    ("2_conversion_test_8b_64tok", ["python", "conversion_test.py", "--model_id", MODEL_8B, "--max_new_tokens", MAXTOK]),
    ("3_aggregate_text_only",      ["python", "text_only_v2.py", "--aggregate", "--models", "internvl3-8b"]),
    ("4_aggregate_conversion",     ["python", "conversion_test.py", "--aggregate", "--models", "internvl3-8b"]),
    ("5_cap_check_8b",             ["python", "cap_check.py", "--model_id", MODEL_8B, "--cap", MAXTOK,
                                    "--csv", "results/internvl3-8b/text_only_v2.csv:raw_output",
                                            "results/internvl3-8b/conversion_test.csv:raw_output",
                                    "--out_csv", "results/internvl3-8b/cap_check_64.csv"]),
    ("6_probe_internvl3_2b",       ["python", "probe.py", "--stage", "all", "--model_id", MODEL_2B, "--adapter", "internvl3-2b",
                                    "--data_csv", "data_balanced/data.csv", "--images_dir", "data_balanced"]),
]
for name, cmd in STEPS:
    print(f"{name:<30} {' '.join(cmd)}")


In [ ]:
# CELL 5 -- RUN every step. A failing step is recorded and the notebook CONTINUES with the next one.
EXPECTED = ['results/internvl3-8b/text_only_v2.csv', 'results/internvl3-8b/run_config.json', 'results/internvl3-8b/conversion_test.csv', 'results/internvl3-8b/conversion_test_run_config.json', 'results/internvl3-8b/cap_check_64.csv', 'probe_output/probe_results/internvl3-2b/per_layer_results.csv', 'probe_output/probe_results/internvl3-2b/shuffled_label_control.csv', 'probe_output/probe_results/internvl3-2b/summary.txt', 'probe_output/activations/internvl3-2b/index.csv']        # files that should exist at the end (relative to the run folder)
status = []

def make_zip():
    """Whitelisted snapshot of everything worth keeping; refreshed after EVERY step."""
    tmp = OUT_ZIP + ".tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for sub in ("results", "logs", "probe_output"):
            for dp, _, fns in os.walk(f"{RUN}/{sub}"):
                for fn in fns:
                    full = os.path.join(dp, fn); z.write(full, os.path.relpath(full, RUN))
        for rel in ["status.json", "STATUS.txt", "env.txt", "conversion_images/manifest.csv",
                    "data_balanced/data.csv"] + CODE_FILES:
            if os.path.exists(f"{RUN}/{rel}"):
                z.write(f"{RUN}/{rel}", rel)
    os.replace(tmp, OUT_ZIP)

def run_step(name, cmd):
    elapsed_h = (time.time() - T_START) / 3600
    if elapsed_h > DEADLINE_H:
        return {"step": name, "status": "SKIPPED (time budget)", "returncode": None, "minutes": 0.0}
    cmd = [sys.executable if c == "python" else c for c in cmd]
    print(f"\n{'=' * 100}\n[{datetime.datetime.utcnow():%H:%M:%S}] START {name}  (elapsed {elapsed_h:.2f} h)\n{'=' * 100}", flush=True)
    t0 = time.time()
    with open(f"{RUN}/logs/{name}.log", "w", encoding="utf-8") as log:
        p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             encoding="utf-8", errors="replace", bufsize=1)
        for line in p.stdout:
            print(line, end="", flush=True); log.write(line)
        rc = p.wait()
    mins = (time.time() - t0) / 60
    print(f"[{name}] finished rc={rc} in {mins:.1f} min", flush=True)
    return {"step": name, "status": "OK" if rc == 0 else f"FAILED rc={rc}", "returncode": rc, "minutes": round(mins, 1)}

for name, cmd in STEPS:
    try:
        status.append(run_step(name, cmd))
    except Exception as e:           # never let one step's crash stop the notebook
        status.append({"step": name, "status": f"ERROR {type(e).__name__}: {e}"[:120], "returncode": None, "minutes": 0.0})
        print(f"[{name}] notebook-side error: {e}", flush=True)
    json.dump(status, open(f"{RUN}/status.json", "w"), indent=1)
    make_zip()                       # insurance: the zip is valid after every step
    print(f"zip refreshed after {name}: {os.path.getsize(OUT_ZIP) / 1e6:.1f} MB", flush=True)


In [ ]:
# CELL 6 -- FINAL: status table, completeness check, ONE output zip
lines = [f"{s['step']:<30} {s['status']:<26} {s['minutes']:>7} min" for s in status]
missing = [f for f in EXPECTED if not os.path.exists(f"{RUN}/{f}")]
lines += ["", f"total wall time: {(time.time() - T_START) / 3600:.2f} h",
          "MISSING expected files: " + (", ".join(missing) if missing else "none")]
open(f"{RUN}/STATUS.txt", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
make_zip()
with zipfile.ZipFile(OUT_ZIP) as z:
    assert z.testzip() is None, "zip is corrupt"
    names = z.namelist()
print(f"\nOUTPUT: {OUT_ZIP}  ({os.path.getsize(OUT_ZIP) / 1e6:.1f} MB, {len(names)} files)")
for n in sorted(names):
    if n.startswith("results/") and n.endswith((".csv", ".json", ".png")):
        print("  ", n)
shutil.rmtree(RUN)      # leave ONLY the zip in /kaggle/working (the zip is verified above)
print("\nDONE." if not missing and all(s["status"] == "OK" for s in status) else
      "\nDONE WITH PROBLEMS -- read STATUS.txt inside the zip and the logs/ folder.")
